# Filtered Area Analysis EDA

This notebook analyzes:

- `classify_output/predictions.csv`
- `classify_output/filtered_areas.csv`

Goals:

- Validate output quality (missing values, duplicates, join consistency).
- Summarize prediction outcomes (`good` / `bad` / `rejected`, if available).
- Analyze filtered-area distributions (global + per-image).
- Detect and inspect outliers in area metrics.
- Provide a final concise **Key Findings** summary.


## 1) Configuration


In [ ]:
from pathlib import Path

# Input files (override if needed)
PREDICTIONS_CSV = Path("../classify_output/predictions.csv")
FILTERED_AREAS_CSV = Path("../classify_output/filtered_areas.csv")

# Analysis controls
TOP_N_IMAGES = 20
TOP_N_OUTLIERS = 20
RANDOM_SEED = 42

# Plot controls
FIG_DPI = 120


## 2) Imports and Plot Style


In [ ]:
from __future__ import annotations

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

print(f"Using seaborn: {HAS_SEABORN}")


## 3) Helper Functions


In [ ]:
def ensure_file(path: Path) -> Path:
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"File not found: {p.resolve()}")
    return p


def resolve_join_keys(pred_df: pd.DataFrame, area_df: pd.DataFrame) -> list[str]:
    preferred = ["image_path", "mask_index"]
    keys = [k for k in preferred if k in pred_df.columns and k in area_df.columns]
    if keys:
        return keys
    if "mask_index" in pred_df.columns and "mask_index" in area_df.columns:
        return ["mask_index"]
    return []


def numeric_cols(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]


def summarize_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    if not cols:
        return pd.DataFrame()
    return df[cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T


def iqr_outlier_mask(s: pd.Series) -> tuple[pd.Series, float, float]:
    s = pd.to_numeric(s, errors="coerce")
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    return (s < lo) | (s > hi), float(lo), float(hi)


def has_col(df: pd.DataFrame, col: str) -> bool:
    return col in df.columns


## 4) Load Data


In [ ]:
pred_path = ensure_file(PREDICTIONS_CSV)
area_path = ensure_file(FILTERED_AREAS_CSV)

pred_df = pd.read_csv(pred_path)
area_df = pd.read_csv(area_path)

print(f"Predictions path: {pred_path.resolve()}")
print(f"Filtered areas path: {area_path.resolve()}")
print(f"pred_df shape: {pred_df.shape}")
print(f"area_df shape: {area_df.shape}")

print("\nPredictions columns:")
print(list(pred_df.columns))
print("\nFiltered areas columns:")
print(list(area_df.columns))

display(pred_df.head())
display(area_df.head())


## 5) Quality Checks (Missing, Duplicates, Basic Validity)


In [ ]:
join_keys = resolve_join_keys(pred_df, area_df)
print(f"Resolved join keys: {join_keys if join_keys else 'None'}")

print("\nMissing values (predictions):")
display(pred_df.isna().sum().to_frame("missing_count").sort_values("missing_count", ascending=False).head(20))

print("\nMissing values (filtered areas):")
display(area_df.isna().sum().to_frame("missing_count").sort_values("missing_count", ascending=False).head(20))

if join_keys:
    pred_dups = int(pred_df.duplicated(subset=join_keys).sum())
    area_dups = int(area_df.duplicated(subset=join_keys).sum())
    print(f"\nDuplicate rows by join keys in predictions: {pred_dups}")
    print(f"Duplicate rows by join keys in filtered areas: {area_dups}")
else:
    print("\nNo common join keys found; duplicate-by-key check skipped.")

area_metric_candidates = ["area_px", "area_um2", "major_axis_px", "minor_axis_px", "major_axis_um", "minor_axis_um"]
for col in area_metric_candidates:
    if has_col(area_df, col):
        invalid = int((pd.to_numeric(area_df[col], errors="coerce") <= 0).sum())
        print(f"Invalid/non-positive values in {col}: {invalid}")

if has_col(pred_df, "predicted_verdict"):
    print("\nPredicted verdict unique values:")
    print(sorted(pred_df["predicted_verdict"].dropna().astype(str).unique().tolist()))

if has_col(pred_df, "accepted"):
    print("\nAccepted value counts:")
    display(pred_df["accepted"].value_counts(dropna=False).rename_axis("accepted").to_frame("count"))


## 6) Predictions EDA (Global and Per-Image)


In [ ]:
if has_col(pred_df, "predicted_verdict"):
    verdict_counts = pred_df["predicted_verdict"].value_counts(dropna=False)
    verdict_pct = (verdict_counts / max(len(pred_df), 1) * 100).round(2)
    verdict_summary = pd.DataFrame({"count": verdict_counts, "percent": verdict_pct})
    print("Predicted verdict distribution:")
    display(verdict_summary)

    plt.figure(figsize=(7, 4), dpi=FIG_DPI)
    colors = ["#009E73", "#D55E00", "#CC79A7", "#4C78A8"]
    verdict_counts.plot(kind="bar", color=colors[: len(verdict_counts)])
    plt.title("Predicted Verdict Distribution")
    plt.xlabel("predicted_verdict")
    plt.ylabel("count")
    plt.tight_layout()
    plt.show()

if has_col(pred_df, "image_path") and has_col(pred_df, "predicted_verdict"):
    per_img = pred_df.groupby("image_path").size().rename("n_cells").sort_values(ascending=False)
    print(f"Top {TOP_N_IMAGES} images by number of predicted cells:")
    display(per_img.head(TOP_N_IMAGES).to_frame())

    ctab = pd.crosstab(pred_df["image_path"], pred_df["predicted_verdict"])
    ctab["total"] = ctab.sum(axis=1)
    ctab = ctab.sort_values("total", ascending=False).head(TOP_N_IMAGES)
    display(ctab)

if has_col(pred_df, "accepted") and has_col(pred_df, "predicted_verdict"):
    print("Accepted x predicted_verdict:")
    display(pd.crosstab(pred_df["predicted_verdict"], pred_df["accepted"], dropna=False))


## 7) Filtered Areas EDA (Global Statistics and Distributions)


In [ ]:
area_numeric = [c for c in numeric_cols(area_df) if c != "mask_index"]
summary = summarize_numeric(area_df, area_numeric)

if summary.empty:
    print("No numeric area columns found in filtered_areas.csv")
else:
    print("Numeric summary for filtered_areas.csv")
    display(summary)

main_metrics = [
    c
    for c in ["area_px", "area_um2", "major_axis_px", "minor_axis_px", "major_axis_um", "minor_axis_um"]
    if c in area_df.columns
]

if main_metrics:
    n = len(main_metrics)
    fig, axes = plt.subplots(n, 2, figsize=(12, max(3 * n, 4)), dpi=FIG_DPI)
    if n == 1:
        axes = np.array([axes])

    for i, col in enumerate(main_metrics):
        s = pd.to_numeric(area_df[col], errors="coerce").dropna()
        ax_h = axes[i, 0]
        ax_b = axes[i, 1]

        ax_h.hist(s, bins=40, color="#1f77b4", alpha=0.8)
        ax_h.set_title(f"Histogram: {col}")
        ax_h.set_xlabel(col)
        ax_h.set_ylabel("count")

        ax_b.boxplot(s, vert=False)
        ax_b.set_title(f"Boxplot: {col}")
        ax_b.set_xlabel(col)

    plt.tight_layout()
    plt.show()
else:
    print("No expected area metrics found for distribution plots.")

if has_col(area_df, "image_path"):
    agg_dict = {}
    if has_col(area_df, "mask_index"):
        agg_dict["n_good_cells"] = ("mask_index", "count")
    else:
        first_col = area_df.columns[0]
        agg_dict["n_good_cells"] = (first_col, "count")

    if has_col(area_df, "area_px"):
        agg_dict["area_px_mean"] = ("area_px", "mean")
        agg_dict["area_px_median"] = ("area_px", "median")

    per_img_area = area_df.groupby("image_path").agg(**agg_dict).sort_values("n_good_cells", ascending=False)

    print(f"Top {TOP_N_IMAGES} images by good-cell count in filtered_areas.csv")
    display(per_img_area.head(TOP_N_IMAGES))


## 8) Join Analysis: `predictions.csv` vs `filtered_areas.csv`


In [ ]:
if not join_keys:
    merged = None
    print("Join skipped: no common keys between predictions and filtered areas.")
else:
    merged = pred_df.merge(area_df, on=join_keys, how="left", indicator=True, suffixes=("", "_area"))

    merge_counts = merged["_merge"].value_counts(dropna=False).rename_axis("merge_flag").to_frame("count")
    merge_counts["percent"] = (merge_counts["count"] / len(merged) * 100).round(2)

    print("Merge coverage (predictions LEFT JOIN filtered_areas):")
    display(merge_counts)

    if has_col(merged, "predicted_verdict"):
        area_present = merged["_merge"].eq("both")
        cov_by_verdict = (
            merged.assign(area_present=area_present)
            .groupby("predicted_verdict")["area_present"]
            .agg(["count", "mean"])
            .rename(columns={"mean": "area_match_rate"})
        )
        cov_by_verdict["area_match_rate"] = (cov_by_verdict["area_match_rate"] * 100).round(2)
        print("Area match rate by predicted_verdict (%):")
        display(cov_by_verdict)

    if has_col(merged, "accepted"):
        area_present = merged["_merge"].eq("both")
        cov_by_acc = (
            merged.assign(area_present=area_present)
            .groupby("accepted")["area_present"]
            .agg(["count", "mean"])
            .rename(columns={"mean": "area_match_rate"})
        )
        cov_by_acc["area_match_rate"] = (cov_by_acc["area_match_rate"] * 100).round(2)
        print("Area match rate by accepted (%):")
        display(cov_by_acc)

    if has_col(merged, "predicted_verdict") and has_col(merged, "area_px"):
        plt.figure(figsize=(8, 4), dpi=FIG_DPI)
        plot_df = merged.dropna(subset=["area_px", "predicted_verdict"]).copy()
        if not plot_df.empty:
            if HAS_SEABORN:
                order = sorted(plot_df["predicted_verdict"].astype(str).unique())
                sns.boxplot(data=plot_df, x="predicted_verdict", y="area_px", order=order)
            else:
                groups = [g["area_px"].dropna().values for _, g in plot_df.groupby("predicted_verdict")]
                labels = [str(k) for k, _ in plot_df.groupby("predicted_verdict")]
                plt.boxplot(groups, labels=labels)
            plt.title("Area Distribution by Predicted Verdict (joined rows)")
            plt.xlabel("predicted_verdict")
            plt.ylabel("area_px")
            plt.tight_layout()
            plt.show()
        else:
            print("No joined area rows available for verdict-wise area boxplot.")


## 9) Outlier Analysis (IQR)


In [ ]:
outlier_reports = {}

candidate_metrics = [
    c
    for c in ["area_px", "area_um2", "major_axis_px", "minor_axis_px", "major_axis_um", "minor_axis_um"]
    if c in area_df.columns
]

if not candidate_metrics:
    print("No candidate metrics found for outlier analysis.")
else:
    for metric in candidate_metrics:
        mask, lo, hi = iqr_outlier_mask(area_df[metric])
        out_df = area_df.loc[mask].copy().sort_values(metric, ascending=False)
        outlier_reports[metric] = {
            "count": int(mask.sum()),
            "low_bound": lo,
            "high_bound": hi,
            "top": out_df.head(TOP_N_OUTLIERS),
        }

    summary_rows = []
    for metric, rep in outlier_reports.items():
        summary_rows.append(
            {
                "metric": metric,
                "outlier_count": rep["count"],
                "low_bound": rep["low_bound"],
                "high_bound": rep["high_bound"],
            }
        )

    outlier_summary = pd.DataFrame(summary_rows).sort_values("outlier_count", ascending=False)
    print("Outlier summary (IQR method):")
    display(outlier_summary)

    for metric, rep in outlier_reports.items():
        print(
            f"\nTop outliers for {metric} (n={rep['count']}, bounds=[{rep['low_bound']:.3f}, {rep['high_bound']:.3f}]):"
        )
        cols = [
            c
            for c in ["image_path", "mask_index", metric, "major_axis_px", "minor_axis_px", "area_um2"]
            if c in rep["top"].columns
        ]
        display(rep["top"][cols])


## 10) Per-Image Deep Dive


In [ ]:
if has_col(pred_df, "image_path") and has_col(pred_df, "predicted_verdict"):
    per_img_pred = pd.crosstab(pred_df["image_path"], pred_df["predicted_verdict"])
    per_img_pred["total"] = per_img_pred.sum(axis=1)
    per_img_pred = per_img_pred.sort_values("total", ascending=False)

    top_img_pred = per_img_pred.head(TOP_N_IMAGES)
    display(top_img_pred)

    stack_cols = [c for c in ["good", "bad", "rejected"] if c in top_img_pred.columns]
    if stack_cols:
        plt.figure(figsize=(12, 5), dpi=FIG_DPI)
        x = np.arange(len(top_img_pred))
        bottom = np.zeros(len(top_img_pred))
        color_map = {"good": "#009E73", "bad": "#D55E00", "rejected": "#CC79A7"}

        for col in stack_cols:
            vals = top_img_pred[col].to_numpy()
            plt.bar(x, vals, bottom=bottom, color=color_map.get(col, None), label=col)
            bottom += vals

        plt.xticks(x, top_img_pred.index, rotation=90)
        plt.ylabel("cell count")
        plt.title(f"Top {TOP_N_IMAGES} Images: Verdict Composition")
        plt.legend()
        plt.tight_layout()
        plt.show()

if has_col(area_df, "image_path") and has_col(area_df, "area_px"):
    per_img_area = area_df.groupby("image_path").agg(
        n_good_cells=("area_px", "count"),
        area_px_mean=("area_px", "mean"),
        area_px_median=("area_px", "median"),
    ).sort_values("n_good_cells", ascending=False)

    top_img_area = per_img_area.head(TOP_N_IMAGES)
    display(top_img_area)

    plt.figure(figsize=(12, 5), dpi=FIG_DPI)
    plt.bar(top_img_area.index, top_img_area["n_good_cells"], color="#4C78A8")
    plt.xticks(rotation=90)
    plt.ylabel("good-cell count")
    plt.title(f"Top {TOP_N_IMAGES} Images by Good Cells (filtered_areas)")
    plt.tight_layout()
    plt.show()


## 11) Final Key Findings


In [ ]:
lines = []

n_pred = len(pred_df)
n_area = len(area_df)
lines.append(f"- Loaded **{n_pred}** prediction rows and **{n_area}** filtered-area rows.")

if has_col(pred_df, "predicted_verdict"):
    vc = pred_df["predicted_verdict"].value_counts(dropna=False)
    parts = [f"{k}: {int(v)}" for k, v in vc.items()]
    lines.append("- Verdict distribution: " + ", ".join(parts) + ".")

if has_col(pred_df, "accepted"):
    acc_rate = pd.to_numeric(pred_df["accepted"], errors="coerce").mean()
    if pd.notna(acc_rate):
        lines.append(f"- Accepted-rate estimate: **{acc_rate * 100:.2f}%**.")

if join_keys:
    merged_tmp = pred_df.merge(area_df, on=join_keys, how="left", indicator=True)
    match_rate = (merged_tmp["_merge"] == "both").mean()
    lines.append(f"- Prediction-to-area join match rate (LEFT JOIN on {join_keys}): **{match_rate * 100:.2f}%**.")
else:
    lines.append("- No common join keys were available between predictions and filtered areas.")

if has_col(area_df, "area_px"):
    area_series = pd.to_numeric(area_df["area_px"], errors="coerce")
    med = area_series.median()
    q1 = area_series.quantile(0.25)
    q3 = area_series.quantile(0.75)
    lines.append(f"- `area_px` median: **{med:.2f}** (IQR: {q1:.2f} to {q3:.2f}).")

if "outlier_reports" in globals() and outlier_reports:
    metric_max = max(outlier_reports.items(), key=lambda kv: kv[1]["count"])
    lines.append(
        f"- Highest outlier count observed in **{metric_max[0]}**: {metric_max[1]['count']} rows (IQR method)."
    )

md = "## Key Findings\n" + "\n".join(lines)
display(Markdown(md))
